# Clip-quality gate for consistent ID (MEVID-SC)

**Finding from the LookFactors study:** CSCI's biggest cause of ID splits is clips where the person is not visible (the box shows grass, a wall or pavement, or the person is hidden). That is 122 of 1,368 clips, affecting 34 of 54 people. Each one becomes a false new ID.

**Method (no labels, no training, CSCI unchanged):**
1. **Quality per clip:** the share of frames where a pose model finds the person, measured in the LookFactors notebook.
2. **Flag:** clips with quality below 0.5 are flagged. This threshold is fixed in advance.
3. **Build IDs from the good clips only** (average linkage).
4. **Flagged clips:** a flagged clip joins its closest ID only if it is as close as the clustering itself allows. Otherwise it stays **unresolved** ("not enough evidence") instead of becoming a false new ID.

**Fair tests:**
- The cut is calibrated on one group of people and tested on the other, at 90% target pair precision.
- **Controls flag the same number of clips:**
  - CSCI's own confidence, i.e. the clips least similar to anything;
  - random clips.
- Results come from the frozen split plus 50 random tests.
- **IDF1 all clips** counts unresolved clips as their own IDs. It is the strict number that nothing can hide from.

**Needs:** the LookFactors results on Drive. No GPU needed. Runtime is about 10 to 15 minutes.

In [ ]:
# 1. Setup: fingerprints (CSCI, the OSNet baseline), the per-clip measurements
#    from the LookFactors notebook, and the frozen two-group protocol.
#    Everything is fixed here before any result is seen.
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

CS = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
                           'test_templates_csci_sc.npz',
                           f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz']))
TR = os.path.dirname(find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
                           'test_templates_tracker_sc.npz',
                           f'{SD}/**/Tracker-SameClothes/test_templates_tracker_sc.npz']))
LF = os.path.dirname(find([f'{SD}/Data/MEVID-results/LookFactors-SameClothes/'
                           'clip_measurements.csv',
                           f'{SD}/**/LookFactors-SameClothes/clip_measurements.csv']))
OUT = os.path.join(os.path.dirname(CS), 'ClipQuality-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True),
                           1e-8)).astype(np.float32)


te = np.load(f'{CS}/test_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
pid, cam = te['pid'], te['cam']
assert (tte['pid'] == pid).all()
FEATS = {'CSCI': unit(te['templates']),
         'OSNet baseline (OSNet)': unit(tte['zs'])}
CL = pd.read_csv(f'{LF}/clip_measurements.csv', index_col=0)
assert len(CL) == len(pid) and (CL.pid.to_numpy() == pid).all()
FR = pd.read_csv(f'{LF}/frame_measurements.csv')
proto = json.load(open(f'{CS}/PROTOCOL_frozen_sc.json'))
fold = np.array([str(proto['folds'][str(int(p))]) for p in pid])

SETTINGS = {'quality': 'share of sampled frames where the pose model '
                       'finds the person',
            'min_quality': 0.5, 'sensitivity': [0.25, 0.75],
            'precision_target': 0.90, 'n_splits': 25, 'random_draws': 10,
            'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
Q = CL.found.to_numpy()
print(f'{len(pid)} clips | flagged (person visible in <50% of frames): '
      f'{(Q < 0.5).sum()} clips, {len(np.unique(pid[Q < 0.5]))} people '
      f'| results -> {OUT}')


In [ ]:
# 2. The method and the metrics.
#    Clip-quality gate (no labels): clips where the person is visible in
#    fewer than half the frames are FLAGGED. Flagged clips cannot start a
#    new ID: IDs are built from the good clips only (average linkage); a
#    flagged clip joins the closest ID only if it is as close as any merge
#    the clustering itself allows (mean distance <= the cut); otherwise it
#    stays UNRESOLVED ("not enough evidence"), instead of a false new ID.
#    Controls flag the SAME NUMBER of clips, chosen (a) by CSCI's own
#    confidence (clips least similar to any other clip) or (b) at random,
#    so any gain must come from WHICH clips the pose check flags.
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment

CUTS = np.linspace(0.01, 1.5, 150)


def _cont(pred, p):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1))
    np.add.at(M, (pi, ci), 1)
    return M, pi, ci


def score(pred, p, c):
    M, pi, ci = _cont(pred, p)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0)
    tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    K = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(K, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (K.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (K ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}


class Group:
    """one group of clips; linkage of the good clips is computed once"""
    def __init__(self, X, idx, flagged):
        self.X, self.idx = X, idx
        self.good = idx[~flagged]
        self.bad = idx[flagged]
        Xg = X[self.good]
        D = np.clip(1 - Xg @ Xg.T, 0, None); D = (D + D.T) / 2
        np.fill_diagonal(D, 0)
        self.L = linkage(squareform(D, checks=False), 'average')
        self.Db = np.clip(1 - X[self.bad] @ Xg.T, 0, None)

    def ids(self, cut):
        lab = fcluster(self.L, cut, 'distance') - 1
        pred = pd.Series(-1, index=self.idx)
        pred[self.good] = lab
        if len(self.bad):
            H = np.zeros((len(self.good), lab.max() + 1)); H[np.arange(len(lab)), lab] = 1
            M = (self.Db @ H) / H.sum(0)
            ok = M.min(1) <= cut
            pred[self.bad[ok]] = M.argmin(1)[ok]
        return pred


def evaluate(pred):
    """scores on the clips that got an ID; unresolved clips are reported
    as coverage, and also counted as one-clip IDs in 'IDF1 all clips'"""
    v = pred.to_numpy(); i = pred.index.to_numpy()
    res = v >= 0
    s = score(v[res], pid[i[res]], cam[i[res]])
    full = v.copy(); full[~res] = v.max() + 1 + np.arange((~res).sum())
    s['IDF1 all clips %'] = score(full, pid[i], cam[i])['IDF1 %']
    s['clips with an ID %'] = 100 * res.mean()
    return s


def calibrate(g, target):
    best = CUTS[0]
    for t in CUTS:
        pred = g.ids(t)
        v = pred.to_numpy(); res = v >= 0
        if score(v[res], pid[pred.index[res]], cam[pred.index[res]]
                 )['pair precision %'] >= 100 * target:
            best = t
        else:
            break
    return best


def flag_sets(X, idx, k_from_quality, rng, controls=True):
    """label-free flagged sets for one group of clips"""
    S = X[idx] @ X[idx].T; np.fill_diagonal(S, -np.inf)
    own = np.argsort(S.max(1))                      # least similar first
    k = int(k_from_quality.sum())
    out = {'CSCI alone (no gate)': np.zeros(len(idx), bool),
           'Clip-quality gate (ours)': k_from_quality}
    if not controls:
        return out
    f = np.zeros(len(idx), bool); f[own[:k]] = True
    out['Control: own-confidence gate'] = f
    for d in range(SETTINGS['random_draws']):
        f = np.zeros(len(idx), bool)
        f[rng.choice(len(idx), k, replace=False)] = True
        out[f'Control: random gate #{d}'] = f
    return out


def run_split(groups, X, min_q, rng, controls=True):
    """groups: dict name -> clip indices. Calibrate the cut on one group,
    test on the other, both directions; returns one row per method x dir."""
    rows = []
    names = list(groups)
    for cal, tst in [(names[0], names[1]), (names[1], names[0])]:
        fc = flag_sets(X, groups[cal], Q[groups[cal]] < min_q, rng, controls)
        ft = flag_sets(X, groups[tst], Q[groups[tst]] < min_q, rng, controls)
        for m in fc:
            gc = Group(X, groups[cal], fc[m])
            cut = calibrate(gc, SETTINGS['precision_target'])
            gt = Group(X, groups[tst], ft[m])
            r = evaluate(gt.ids(cut))
            r.update(method=m.split(' #')[0], direction=f'{cal}->{tst}')
            rows.append(r)
    return pd.DataFrame(rows).groupby(['method', 'direction'],
                                      sort=False).mean(numeric_only=True)
print('ready')


In [ ]:
# 3. Frozen protocol (groups A and B): calibrate on one, test on the
#    other, both directions, averaged.
rng = np.random.default_rng(SETTINGS['seed'])
G = {g: np.flatnonzero(fold == g) for g in sorted(np.unique(fold))}
COLS = ['clips with an ID %', 'IDs per person', 'IDF1 %', 'pair precision %',
        'cross-camera recall %', 'IDF1 all clips %']
MAIN = {}
for bb, X in FEATS.items():
    t0 = time.time()
    R = run_split(G, X, SETTINGS['min_quality'], rng)
    MAIN[bb] = R.groupby(level=0, sort=False).mean()[COLS].round(2)
    print(f'\n=== {bb}  (target pair precision 90%, mean of both '
          f'directions, {time.time() - t0:.0f} s) ===')
    display(MAIN[bb])
    MAIN[bb].to_csv(f'{OUT}/main_{bb.split()[0]}.csv')


In [ ]:
# 4. Robustness: 25 random person splits x 2 directions = 50 tests.
#    Paired comparison of the gate against CSCI alone and the controls.
from tqdm.auto import tqdm
rng = np.random.default_rng(SETTINGS['seed'] + 1)
people = np.unique(pid)
REP = []
for s in tqdm(range(SETTINGS['n_splits']), desc='splits'):
    perm = rng.permutation(people)
    a = np.isin(pid, perm[:len(people) // 2])
    groups = {'A': np.flatnonzero(a), 'B': np.flatnonzero(~a)}
    for bb, X in FEATS.items():
        for mq in [SETTINGS['min_quality']] + SETTINGS['sensitivity']:
            R = run_split(groups, X, mq, rng,
                          controls=mq == SETTINGS['min_quality']).reset_index()
            R['backbone'], R['split'], R['min_quality'] = bb, s, mq
            REP.append(R)
REP = pd.concat(REP)
REP.to_csv(f'{OUT}/robustness_all.csv', index=False)
main = REP[REP.min_quality == SETTINGS['min_quality']]
summ = main.groupby(['backbone', 'method'], sort=False)[COLS].agg(
    ['mean', 'std']).round(2)
print('Mean (sd) over 50 tests, quality threshold 0.5:')
display(summ)

key = ['backbone', 'split', 'direction']
piv = main.pivot_table(index=key, columns='method',
                       values=['IDs per person', 'IDF1 %', 'pair precision %'])
OURS = 'Clip-quality gate (ours)'
PAIR = []
for bb in FEATS:
    p = piv.xs(bb, level='backbone')
    for other in ['CSCI alone (no gate)', 'Control: own-confidence gate',
                  'Control: random gate']:
        for metric, better in [('IDs per person', -1), ('IDF1 %', 1),
                               ('pair precision %', 1)]:
            d = p[(metric, OURS)] - p[(metric, other)]
            PAIR.append({'backbone': bb, 'ours vs': other, 'metric': metric,
                         'mean difference': round(d.mean(), 2),
                         '95% range of difference':
                             f'[{np.percentile(d, 2.5):.2f}, '
                             f'{np.percentile(d, 97.5):.2f}]',
                         'ours better in % of tests':
                             round(100 * (better * d > 0).mean(), 0)})
PAIR = pd.DataFrame(PAIR)
PAIR.to_csv(f'{OUT}/paired_comparison.csv', index=False)
print('\nPaired comparison (50 tests):')
display(PAIR)
sens = REP[REP.method == OURS].groupby(['backbone', 'min_quality'])[
    COLS].mean().round(2)
print('\nSensitivity to the quality threshold (fixed in advance):')
display(sens)


In [ ]:
# 5. What do the flagged clips look like inside? This decides the next
#    step for occlusion: if most flagged clips still contain SOME frames
#    with the person, rebuilding the fingerprint from those frames only
#    can rescue them; if they are empty throughout, they can only be
#    flagged.
fl = np.flatnonzero(Q < SETTINGS['min_quality'])
kind = pd.cut(Q[fl], [-0.01, 0.0, 0.25, 0.4999],
              labels=['no frame with the person',
                      'person in 1-25% of frames',
                      'person in 25-50% of frames'])
A = pd.DataFrame({'clip': fl, 'person': pid[fl], 'camera': cam[fl],
                  'quality': Q[fl], 'kind': kind,
                  'frames': CL.frames.to_numpy()[fl]})
A.to_csv(f'{OUT}/flagged_clips.csv', index=False)
print(f'{len(fl)} flagged clips ({100 * len(fl) / len(pid):.1f}% of all), '
      f'{A.person.nunique()} people, {A.camera.nunique()} cameras')
display(A.kind.value_counts().to_frame('clips'))
print('\nFlagged clips per camera (share of that camera\'s clips):')
display((pd.Series(cam[fl]).value_counts() / pd.Series(cam).value_counts()
         ).dropna().sort_values(ascending=False).mul(100).round(1)
        .to_frame('% flagged'))
import shutil
shutil.make_archive('/content/ClipQuality_results', 'zip', OUT)
print('Saved to', OUT, '\nZip: /content/ClipQuality_results.zip')
